# m06_l04_release

Exobean Engineering. Run the cells from top to bottom.

In [ ]:
# Google Colab only: fetches the free code companion, then run the rest.
import io, os, sys, urllib.request, zipfile
URL = "https://github.com/exobean/ai-agents-level1/releases/latest/download/shopassist-companion.zip"
if "google.colab" in sys.modules and not os.path.isdir("shopassist"):
    try:
        data = io.BytesIO(urllib.request.urlopen(URL).read())
    except OSError:  # network blocks the download: upload the zip
        from google.colab import files
        data = next(iter(files.upload()))
    zipfile.ZipFile(data).extractall(".")
    os.chdir(next(d for d in os.listdir(".") if d.endswith("-companion")))
    sys.path.insert(0, os.getcwd())


## Release

In [ ]:
import subprocess
import sys
from pathlib import Path

import shopassist
from shopassist.app_v2 import new_session
from shopassist.evals import EVAL_SET, run_eval, wilson

companion = Path(shopassist.__file__).parents[1]

# 1. Every unit and replay test.
tests = subprocess.run([sys.executable, "-m", "pytest", "-q",
                        "-p", "no:cacheprovider", "tests"],
                       cwd=companion, capture_output=True, text=True)
print("tests:", "pass" if tests.returncode == 0 else "FAIL")

# 2. The eval, with a minimum we agreed in advance.
passed = sum(ok for _, ok in run_eval(lambda: new_session()[0]))
low, _ = wilson(passed, len(EVAL_SET))
print(f"eval:  {passed}/{len(EVAL_SET)}, lower bound {low:.0%}")

ready = tests.returncode == 0 and passed == len(EVAL_SET)
print("release ready:", ready)
